# Olist Brazilian E-Commerce: Exploratory Data Analysis & Business Insights

**Objective:** Analyze delivery performance, customer satisfaction (review scores), seller risk factors, geographic bottlenecks, and customer retention using the cleaned dataset created in Phase 4.

---

### Key Business Questions Addressed:
1. **Delivery Delay vs Customer Satisfaction:** How severely does delivery delay impact review scores?
2. **Geographic Bottlenecks:** Which Brazilian states experience the highest delivery delays?
3. **Seller Risk Matrix:** Which high-revenue sellers have unacceptable customer satisfaction ratings?
4. **Product Category Performance:** Which categories lead sales and which suffer high freight / low ratings?
5. **Customer Retention:** What is the overall Repeat Purchase Rate, and does first-order delivery experience influence repeat purchases?

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'Segoe UI, Arial, sans-serif'
plt.rcParams['figure.dpi'] = 120

# Paths
BASE_DIR = '..'
DATA_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'olist_master_analytical_dataset.csv')
ORDERS_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'olist_orders_cleaned.csv')
CUST_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'olist_customers_cleaned.csv')

# Load datasets
df_master = pd.read_csv(DATA_PATH)
df_orders = pd.read_csv(ORDERS_PATH)
df_cust = pd.read_csv(CUST_PATH)

print(f"Loaded Master Dataset: {df_master.shape[0]:,} rows, {df_master.shape[1]} columns.")

## 1. Delivery Delay vs Customer Satisfaction

Let us examine how review scores change across delivery delay buckets (`On Time / Early`, `1-3 Days Late`, `4-7 Days Late`, `> 7 Days Late`).

In [ ]:
# Filter for delivered orders (deduplicate by order_id for order-level metrics)
deliv = df_master[df_master['order_status'] == 'delivered'].drop_duplicates(subset=['order_id']).copy()

bucket_order = ['On Time / Early', '1-3 Days Late', '4-7 Days Late', '> 7 Days Late']
q1_stats = deliv.groupby('delay_bucket').agg(
    order_count=('order_id', 'count'),
    mean_review=('review_score', 'mean'),
    pct_1_star=('review_score', lambda s: (s == 1).mean() * 100),
    pct_5_star=('review_score', lambda s: (s == 5).mean() * 100)
).reindex(bucket_order).reset_index()

display(q1_stats.style.format({
    'order_count': '{:,}',
    'mean_review': '{:.2f} ★',
    'pct_1_star': '{:.2f}%',
    'pct_5_star': '{:.2f}%'
}))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ca02c', '#a1d99b', '#ff7f0e', '#d62728']

# Mean review score
ax1.bar(q1_stats['delay_bucket'], q1_stats['mean_review'], color=colors, width=0.5, edgecolor='black', linewidth=0.5)
ax1.set_title('Average Review Score by Delay Bucket', fontweight='bold')
ax1.set_ylim(0, 5)
for i, v in enumerate(q1_stats['mean_review']):
    ax1.text(i, v + 0.1, f"{v:.2f}", ha='center', fontweight='bold')

# % of 1-Star reviews
ax2.bar(q1_stats['delay_bucket'], q1_stats['pct_1_star'], color=colors, width=0.5, edgecolor='black', linewidth=0.5)
ax2.set_title('% of 1-Star Reviews (Severe Dissatisfaction)', fontweight='bold')
ax2.set_ylim(0, 80)
for i, v in enumerate(q1_stats['pct_1_star']):
    ax2.text(i, v + 1.5, f"{v:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

> **Key Takeaway 1:** There is a clear **"Tolerance Cliff"** at 3 days. A 1-3 day delay reduces the review score slightly (4.29 to 3.76), but from day 4 onwards, customer ratings collapse to 2.32 (4-7 days) and 1.73 (>7 days), with over half of all customers leaving 1-star reviews.

## 2. Geographic Bottlenecks & Interstate Delivery

Which states suffer the most frequent delays, and how does interstate delivery affect transit times?

In [ ]:
state_stats = deliv.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    delay_rate=('is_delayed', lambda s: s.mean() * 100),
    avg_delivery_days=('actual_delivery_days', 'mean'),
    avg_review=('review_score', 'mean')
).reset_index()

# States with >= 100 orders
top_delayed_states = state_stats[state_stats['total_orders'] >= 100].sort_values(by='delay_rate', ascending=False).head(10)

display(top_delayed_states.style.format({
    'total_orders': '{:,}',
    'delay_rate': '{:.2f}%',
    'avg_delivery_days': '{:.1f} days',
    'avg_review': '{:.2f} ★'
}))

In [ ]:
# Interstate comparison
inter_stats = deliv.groupby('is_interstate').agg(
    orders=('order_id', 'count'),
    delay_rate=('is_delayed', lambda s: s.mean() * 100),
    avg_days=('actual_delivery_days', 'mean'),
    avg_review=('review_score', 'mean')
).reset_index()
inter_stats['type'] = inter_stats['is_interstate'].map({0: 'Intrastate (Same State)', 1: 'Interstate (Cross-State)'})
display(inter_stats[['type', 'orders', 'delay_rate', 'avg_days', 'avg_review']])

> **Key Takeaway 2:** Northeast states (`AL`, `MA`, `PI`, `CE`, `SE`) suffer the highest delay rates (15% to 24%). Interstate delivery takes nearly twice as long as intrastate (15.1 days vs 7.9 days).

## 3. Seller Risk Matrix (Revenue vs Customer Satisfaction)

Identify high-revenue sellers whose low ratings threaten platform reputation.

In [ ]:
# Grain-Safe: Deduplicate at (seller_id, order_id) first to preserve order-level grain of reviews and delays
valid_deliv = df_master[(df_master['order_status'] == 'delivered') & (df_master['is_valid_delivered'] == 1)].copy()

seller_orders = valid_deliv.groupby(['seller_id', 'order_id']).agg(
    order_rev=('price', 'sum'),
    items_count=('order_item_id', 'count'),
    review_score=('review_score', 'mean'),
    is_delayed=('is_delayed', 'max')
).reset_index()

seller_stats = seller_orders.groupby('seller_id').agg(
    revenue=('order_rev', 'sum'),
    orders=('order_id', 'count'),
    items_sold=('items_count', 'sum'),
    avg_review=('review_score', 'mean'),
    delay_rate=('is_delayed', lambda s: s.mean() * 100)
).reset_index()

REV_LIMIT = 15000
REVIEW_LIMIT = 3.8

high_risk = seller_stats[(seller_stats['revenue'] >= REV_LIMIT) & (seller_stats['avg_review'] < REVIEW_LIMIT)].sort_values(by='revenue', ascending=False)
print(f"Found {len(high_risk)} High-Risk Sellers (Order-Deduplicated Standard: Revenue >= R$ 15,000 and Review < 3.8).")
display(high_risk.head(10).style.format({
    'revenue': 'R$ {:,.2f}',
    'orders': '{:,}',
    'items_sold': '{:,}',
    'avg_review': '{:.2f} ★',
    'delay_rate': '{:.2f}%'
}))

# Direct item-weighted comparison (yielding 21 sellers)
item_weighted_risk = df_master[df_master['order_status'] == 'delivered'].groupby('seller_id').agg(
    revenue=('price', 'sum'), avg_rev=('review_score', 'mean')
).reset_index()
item_weighted_count = len(item_weighted_risk[(item_weighted_risk['revenue'] >= REV_LIMIT) & (item_weighted_risk['avg_rev'] < REVIEW_LIMIT)])
print(f"Note: Item-weighted standard yields {item_weighted_count} High-Risk sellers.")

> **Key Takeaway 3:** 16 sellers (Order-deduplicated standard) or 21 sellers (Item-weighted standard) account for substantial revenue while generating disproportionate volumes of negative reviews. These sellers should be targeted for dispatch SLA compliance.

## 4. Product Category Performance Matrix

Analyze sales, freight cost percentage, and customer satisfaction across top product categories.

In [ ]:
# Full delivered items dataset (avoiding dropped duplicate items)
deliv_items = df_master[df_master['order_status'] == 'delivered'].copy()
cat_stats = deliv_items.groupby('product_category_formatted').agg(
    revenue=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    total_item_value=('item_total_value', 'sum'),
    items_sold=('order_item_id', 'count'),
    avg_review=('review_score', 'mean'),
    delay_rate=('is_delayed', lambda s: s.mean() * 100)
).reset_index()

cat_stats['freight_pct'] = ((cat_stats['total_freight'] / cat_stats['total_item_value']) * 100).round(2)
cat_stats = cat_stats.sort_values(by='revenue', ascending=False)

display(cat_stats.head(12).style.format({
    'revenue': 'R$ {:,.2f}',
    'total_freight': 'R$ {:,.2f}',
    'freight_pct': '{:.1f}%',
    'items_sold': '{:,}',
    'avg_review': '{:.2f} ★',
    'delay_rate': '{:.2f}%'
}))

## 5. Customer Retention & First Order Impact

Measure Repeat Purchase Rate and evaluate whether delivery delay on the first order reduces the likelihood of return purchases.

In [ ]:
merged = df_orders.merge(df_cust[['customer_id', 'customer_unique_id']], on='customer_id')

# 1. All orders repeat rate
all_order_counts = merged.groupby('customer_unique_id')['order_id'].nunique()
total_customers = len(all_order_counts)
repeat_customers = (all_order_counts > 1).sum()
repeat_rate = (repeat_customers / total_customers) * 100
print(f"Total Unique Customers (All Statuses): {total_customers:,} | Repeat: {repeat_customers:,} ({repeat_rate:.2f}%)")

# 2. Delivered completed repeat rate
deliv_orders = merged[merged['order_status'] == 'delivered'].copy()
deliv_order_counts = deliv_orders.groupby('customer_unique_id')['order_id'].nunique()
total_deliv_cust = len(deliv_order_counts)
repeat_deliv_cust = (deliv_order_counts > 1).sum()
repeat_deliv_rate = (repeat_deliv_cust / total_deliv_cust) * 100
repeat_deliv_ids = set(deliv_order_counts[deliv_order_counts > 1].index)
print(f"Total Unique Customers (Delivered Only): {total_deliv_cust:,} | Repeat Completed: {repeat_deliv_cust:,} ({repeat_deliv_rate:.2f}%)")

# 3. First order delivery impact on completed repeat purchase
deliv_sorted = deliv_orders.sort_values(by=['customer_unique_id', 'order_purchase_timestamp'])
deliv_first = deliv_sorted.drop_duplicates(subset=['customer_unique_id'], keep='first').copy()
deliv_first = deliv_first[deliv_first['is_valid_delivered'] == 1].copy()
deliv_first['is_repeat'] = deliv_first['customer_unique_id'].isin(repeat_deliv_ids).astype(int)

retention_impact = deliv_first.groupby('is_delayed').agg(
    customers=('customer_unique_id', 'count'),
    repeat_purchase_rate=('is_repeat', lambda s: s.mean() * 100)
).reset_index()
retention_impact['first_order_status'] = retention_impact['is_delayed'].map({0: '1st Order On-Time', 1: '1st Order Delayed'})

display(retention_impact[['first_order_status', 'customers', 'repeat_purchase_rate']])

> **Key Takeaway 5:** Marketplace repeat rate is **3.12%** across all orders (and **3.00%** for completed orders). Customers who experienced a delay on their first order are **17.4% less likely to return** (repeat purchase rate falls from 3.04% to 2.51%).